<a href="https://colab.research.google.com/github/amina-nasrin/Graph-Neural-Network/blob/main/Assignment3_Nasrin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install wget

# New Section

In [ ]:
import wget
import zipfile
import os

url = 'https://snap.stanford.edu/data/congress_network.zip'
wget.download(url)

with zipfile.ZipFile("congress_network.zip", 'r') as zip_ref:
  zip_ref.extractall("congress_network")

extracted_files = os.listdir("congress_network")

In [ ]:
import pandas as pd
df = pd.read_csv("congress_network/congress_network/congress.edgelist", delim_whitespace=True, header=None, names=['Source', 'Target', 'weight:', 'Weight'])

In [ ]:
df2 = df[['Source', 'Target']]
df2.head(5)

In [ ]:
!pip install torch_geometric

In [ ]:
import torch
from torch_geometric.data import Data
import torch_geometric.utils as pyg_utils

edge_index = torch.tensor(df2.values.T, dtype=torch.long)
data = Data(edge_index=edge_index)
print(f'Self-Loops: {pyg_utils.contains_self_loops(data.edge_index)}')

In [ ]:
df2['Source'] = pd.to_numeric(df2['Source'], errors='coerce')
df2['Target'] = pd.to_numeric(df2['Target'], errors='coerce')

df2.dropna(inplace=True)

df2 = df2.astype(int)
data.edge_index = torch.tensor(df2.values.T, dtype=torch.long)

if data.edge_index.size(1)<edge_index.size(1):
  print("Duplicate")


In [ ]:
import networkx as nx
import numpy as np

G = nx.from_pandas_edgelist(df2, source='Source', target='Target', create_using=nx.DiGraph())
n = G.number_of_nodes()

alpha_katz = 1 / n
beta_katz = 1

A = nx.to_numpy_array(G)

I = np.eye(n)

ones = np.ones((n, 1))

katz_centrality = (
    I
    + alpha_katz * A  @ ones
    + alpha_katz**2 * np.linalg.matrix_power(A, 2) @ ones
    + alpha_katz**3 * np.linalg.matrix_power(A, 3) @ ones
) * beta_katz

katz_centrality_dict = {node: katz_centrality[i, 0] for i, node in enumerate(G.nodes())}

katz_sample = {k: katz_centrality_dict[k] for k in list(katz_centrality_dict)[:5]}
print("Katz Centrality:", katz_centrality)

katz_centrality_nx = nx.katz_centrality(G, alpha=alpha_katz, beta=beta_katz)
for node, centrality in katz_centrality_nx.items():
    print(f"Node {node}: {centrality}")



In [ ]:
import matplotlib.pyplot as plt

node_ids = list(katz_centrality_dict.keys())
centrality_values = list(katz_centrality_dict.values())

plt.figure(figsize=(10, 6))
plt.bar(node_ids, centrality_values, color='skyblue', edgecolor='orange')

plt.xlabel("Node IDs", fontsize=20)
plt.ylabel("Katz Centrality Values", fontsize=20)
plt.title("Katz Centrality Measure for Nodes", fontsize=20)
plt.xticks(rotation=90)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.xticks(fontsize=20)
plt.yticks(fontsize=20)
plt.tight_layout()
plt.show()


In [ ]:
nodes = pd.concat([df2['Source'], df2['Target']]).unique()
node_to_idx = {node: idx for idx, node in enumerate(nodes)}
idx_to_node = {idx: node for node, idx in node_to_idx.items()}
N = len(nodes)

A = np.zeros((N, N))
for _, row in df2.iterrows():
    i = node_to_idx[row['Source']]
    j = node_to_idx[row['Target']]
    A[j, i] = 1

out_degrees = A.sum(axis=0)

alpha = 0.85
x = np.ones(N) / N
tolerance = 1e-5
max_iter = 80

for _ in range(max_iter):
    x_new = (1 - alpha) / N + alpha * (A @ (x / np.where(out_degrees > 0, out_degrees, 1)))
    if np.linalg.norm(x_new - x, 1) < tolerance:
        break
    x = x_new

pagerank = {idx_to_node[i]: rank for i, rank in enumerate(x)}

top_10_pagerank = sorted(pagerank.items(), key=lambda item: item[1], reverse=True)[:10]
print("Top 10 nodes by PageRank centrality:")
print(top_10_pagerank)

top_10_pagerank = sorted(pagerank.items(), key=lambda item: item[1], reverse=True)[:10]

print("Top 10 nodes by PageRank centrality (as a list):")
for node, rank in top_10_pagerank:
    print(f"Node: {node}, PageRank: {rank:.6f}")



In [ ]:
nodes, values = zip(*sorted_pagerank)

plt.figure(figsize=(12, 6))
plt.bar(nodes, values, color='skyblue', edgecolor='orange')
plt.xlabel("Node IDs", fontsize=20)
plt.ylabel("PageRank Values", fontsize=20)
plt.title("PageRank Centrality for All Nodes", fontsize=20)
plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
pagerank_x = nx.pagerank(G, alpha=alpha)

top_10_pagerank_x = sorted(pagerank_x.items(), key=lambda x: x[1], reverse=True)[:10]
print("Top 10 nodes by PageRank centrality:")
for node, pr_value_x in top_10_pagerank_x:
    print(f"Node {node}: {pr_value_x}")

In [ ]:
absolute_diff = {}
for node in pagerank:
    if node in pagerank_x:
        absolute_diff[node] = abs(pagerank[node] - pagerank_x[node])

percentage_accuracy = {}
for node, diff in absolute_diff.items():
    if node in pagerank_x:
        accuracy = 100 * (1 - (diff / pagerank_x[node]))
        percentage_accuracy[node] = accuracy


overall_accuracy = np.mean(list(percentage_accuracy.values()))
print(f"\nOverall Percentage Accuracy: {overall_accuracy:.2f}%")


Overall Percentage Accuracy: 99.92%


In [ ]:
import matplotlib.pyplot as plt

manual_node_ids = list(pagerank.keys())
manual_pagerank_values = list(pagerank.values())

nx_node_ids = list(pagerank_x.keys())
nx_pagerank_values = list(pagerank_x.values())

plt.figure(figsize=(12, 6))

plt.bar(manual_node_ids, manual_pagerank_values, width=0.4, label='Manual PageRank', alpha=0.6, align='center')

plt.bar(nx_node_ids, nx_pagerank_values, width=0.4, label='NetworkX PageRank', alpha=0.6, color='red', align='edge')

plt.xlabel("Node IDs", fontsize=20)
plt.ylabel("PageRank Values", fontsize=20)
plt.title("Comparison of PageRank Centrality: Manual vs. NetworkX", fontsize=20)
plt.legend()

plt.xticks(rotation=90, fontsize=20)
plt.yticks(fontsize=20)
plt.grid(True)
plt.tight_layout()
plt.show()
